In [1]:
import pandas as pd
import numpy as np

# --- 1. CARGAMOS LA MUESTRA DE LA METADATA ---
# Asegúrate de que la ruta sea correcta según tu carpeta
ruta_meta = 'meta_Electronics.json/meta_Electronics.json' 

print("Cargando metadata...")
df_meta = pd.read_json(ruta_meta, lines=True, nrows=100000)

print(f"Filas originales: {len(df_meta)}")
print(f"Columnas originales: {len(df_meta.columns)}")

# --- 2. EL ATAJO DEL SENIOR (Quedarnos solo con la carnita) ---
# Solo necesitamos el ID, título, marca, categoría y precio.
columnas_utiles = ['asin', 'title', 'brand', 'category', 'price']
columnas_existentes = [col for col in columnas_utiles if col in df_meta.columns]
df_meta = df_meta[columnas_existentes]

# --- 3. APLANAR LISTAS (El truco mágico para SQL Server) ---
# La categoría viene como lista [A, B, C]. La unimos en un solo texto "A, B, C"
if 'category' in df_meta.columns:
    df_meta['category'] = df_meta['category'].apply(
        lambda x: ', '.join(x) if isinstance(x, list) else "Sin categoría"
    )

# --- 4. LIMPIEZA DE PRECIOS Y NULOS ---
if 'price' in df_meta.columns:
    # Quitamos el signo de dólar y comas
    df_meta['price'] = df_meta['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
    # Convertimos a número. Si hay texto raro (ej. "Ver en el carrito"), lo vuelve NaN, y luego lo pasamos a 0.0
    df_meta['price'] = pd.to_numeric(df_meta['price'], errors='coerce').fillna(0.0)

# El título y la marca no pueden ser nulos, les ponemos texto por defecto
if 'title' in df_meta.columns:
    df_meta['title'] = df_meta['title'].fillna("Sin título")
if 'brand' in df_meta.columns:
    df_meta['brand'] = df_meta['brand'].fillna("Sin marca")

# --- 5. ELIMINAR DUPLICADOS DE PRODUCTOS ---
# Un producto (asin) solo debe existir UNA vez en la metadata
df_meta = df_meta.dropna(subset=['asin'])
df_meta = df_meta.drop_duplicates(subset=['asin'])

print(f"Filas finales listas y limpias: {len(df_meta)}")
print("\n--- METADATA 100% LISTA PARA LA BASE DE DATOS ---")
display(df_meta.head())

Cargando metadata...
Filas originales: 100000
Columnas originales: 19
Filas finales listas y limpias: 69632

--- METADATA 100% LISTA PARA LA BASE DE DATOS ---


,asin,title,brand,category,price
0,0011300000,Genuine Geovision 1 Channel 3rd Party NVR IP S...,GeoVision,"Electronics, Camera &amp; Photo, Video Surveil...",65.00
1,0043396828,"Books ""Handbook of Astronomical Image Processi...",33 Books Co.,"Electronics, Camera &amp; Photo",0.00
2,0060009810,One Hot Summer,Visit Amazon's Carolina Garcia Aguilera Page,"Electronics, eBook Readers &amp; Accessories, ...",11.49
3,0060219602,Hurray for Hattie Rabbit: Story and pictures (...,Visit Amazon's Dick Gackenbach Page,"Electronics, eBook Readers & Accessories, eBoo...",0.00
4,0060786817,sex.lies.murder.fame.: A Novel,Visit Amazon's Lolita Files Page,"Electronics, eBook Readers & Accessories, eBoo...",13.95
